# Cleaning Netflix titles

Cleans the raw `netflix_titles.csv` (movies and TV shows together) and builds a separate countries table, because many titles list several countries.

The result is saved as CSV files in `data/processed/` and loaded into PostgreSQL.
Database settings are read from `.env` (see `.env.example`).

In [1]:
# Importing libraries
import os

import pandas as pd
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

## 1. Load raw data

In [2]:
df_original = pd.read_csv("../data/raw/netflix_titles.csv", encoding="utf-8")
df = df_original.copy()
df["type"].value_counts()

type
Movie      6131
TV Show    2676
Name: count, dtype: int64

## 2. Fix shifted ratings

In a few rows the duration (for example `74 min`) ended up in the `rating` column and `duration` is empty. Move the value back to `duration` and clear the rating. This has to run before the duration is parsed.

In [3]:
mask = df["rating"].str.contains("min", na=False)

df.loc[mask, "duration"] = df.loc[mask, "rating"]
df.loc[mask, "rating"] = None

df.loc[mask, ["title", "type", "rating", "duration"]]

,title,type,rating,duration
5541,Louis C.K. 2017,Movie,None,74 min
5794,Louis C.K.: Hilarious,Movie,None,84 min
5813,Louis C.K.: Live at the Comedy Store,Movie,None,66 min


## 3. Split duration into a number and a unit

Movies are measured in minutes (`90 min`) and TV shows in seasons (`2 Seasons`), so the text is split into `duration_value` and `duration_unit`, matching `sql/01_schema.sql`.

In [4]:
df["duration_value"] = df["duration"].str.extract(r"(\d+)", expand=False).astype("Int64")
df["duration_unit"] = df["type"].map({"Movie": "min", "TV Show": "season"})
df = df.drop(columns="duration")

df.groupby("type")["duration_value"].describe()

,count,mean,std,min,25%,50%,75%,max
type,,,,,,,,
Movie,6131.0,99.564998,28.289504,3.0,87.0,98.0,114.0,312.0
TV Show,2676.0,1.764948,1.582752,1.0,1.0,1.0,2.0,17.0


## 4. Countries table

`country` can hold several countries in one cell (`United States, India`). Split it into one row per title and country, trim spaces and drop empty values. `type` is kept so countries can be compared for movies and TV shows without a join.

In [5]:
df_countries = df[["show_id", "country","type"]].copy()
df_countries["country"] = df_countries["country"].str.split(",")
df_countries = df_countries.explode("country")

df_countries["country"] = df_countries["country"].str.strip()
df_countries = df_countries[df_countries["country"].notna() & (df_countries["country"] != "")]
df_countries = df_countries.drop_duplicates().reset_index(drop=True)

df_countries.head()

,show_id,country,type
0,s1,United States,Movie
1,s2,South Africa,TV Show
2,s5,India,TV Show
3,s8,United States,Movie
4,s8,Ghana,Movie


## 5. Save and load

Cleaned data is saved to `data/processed/`:

- `netflix_titles_full.csv`: all cleaned titles
- `movies.csv` and `tv_shows.csv`: the same data split by `type`
- `countries.csv`: one row per title and country

The same tables are loaded into PostgreSQL using the settings from `.env`. `URL.create` builds the connection string safely, even if the password has special characters. Existing tables are replaced on every run.

In [ ]:
df_movies = df[df["type"] == "Movie"]
df_tv = df[df["type"] == "TV Show"]

out_dir = "../data/processed"
os.makedirs(out_dir, exist_ok=True)

df.to_csv(os.path.join(out_dir, "netflix_titles_full.csv"), index=False)
df_movies.to_csv(os.path.join(out_dir, "movies.csv"), index=False)
df_tv.to_csv(os.path.join(out_dir, "tv_shows.csv"), index=False)
df_countries.to_csv(os.path.join(out_dir, "countries.csv"), index=False)

load_dotenv(find_dotenv(usecwd=True))

url = URL.create(
    "postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT", 5432)),
    database=os.getenv("DB_NAME"),
)
engine = create_engine(url)

df_movies.to_sql("movies", engine, schema="public", if_exists="replace", index=False)
df_tv.to_sql("tv_shows", engine, schema="public", if_exists="replace", index=False)
df_countries.to_sql("countries", engine, schema="public", if_exists="replace", index=False)